In [ ]:
!pip install rembg onnxruntime tqdm opencv-python gradio ultralytics --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import random
from pathlib import Path
from PIL import Image, ImageOps
from tqdm import tqdm
from rembg import remove, new_session

In [ ]:
#Original full dataset in drive
ORIGINAL_DATASET = "/content/drive/MyDrive/thesis/data/asl_alphabet_train/asl_alphabet_train"

SRC_DRIVE = "/content/drive/MyDrive/large_asl_test_set/test_set_8596_second_half"

#Folder containing environment background images
ENV_BG_DIR = "/content/drive/MyDrive/large_asl_test_set/background_images"

#Output folders in Drive
OUT_SOLID_DRIVE = "/content/drive/MyDrive/large_asl_test_set/generated_test_set_solid_bg_fast"
OUT_ENV_DRIVE   = "/content/drive/MyDrive/large_asl_test_set/generated_test_set_env_bg_fast"
OUT_FLIP_DRIVE  = "/content/drive/MyDrive/large_asl_test_set/generated_test_set_hand_flip_fast"

#Temporary local working folder
TMP_ROOT = "/content/tmp_processing"

#Output image size
OUT_SIZE = (224, 224)

#Keep 307 images per class
IMAGES_PER_CLASS = 307

#Solid colors to use
SOLID_COLORS = [
    "#FF0000",  # red
    "#00FF00",  # green
    "#0000FF",  # blue
    "#FFFF00",  # yellow
    "#FFFFFF",  # white
    "#808080",  # gray
]

In [ ]:
if os.path.exists(SRC_DRIVE):
    print(f"Selected dataset already exists at: {SRC_DRIVE}")
else:
    os.makedirs(SRC_DRIVE, exist_ok=True)


    class_dirs = sorted([
        d for d in os.listdir(ORIGINAL_DATASET)
        if os.path.isdir(os.path.join(ORIGINAL_DATASET, d)) and d != "nothing"
    ])

    print("Classes used:", class_dirs)
    print("Total classes:", len(class_dirs))

    print("\nCreating dataset from second half of each class.")
    total_selected = 0

    for cls in tqdm(class_dirs):
        src_class = os.path.join(ORIGINAL_DATASET, cls)
        dst_class = os.path.join(SRC_DRIVE, cls)
        os.makedirs(dst_class, exist_ok=True)

        imgs = sorted([
            f for f in os.listdir(src_class)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        half_idx = len(imgs) // 2
        second_half_imgs = imgs[half_idx:]

        # Keep only 307 images
        selected_imgs = second_half_imgs[:IMAGES_PER_CLASS]

        for img_name in selected_imgs:
            shutil.copy2(
                os.path.join(src_class, img_name),
                os.path.join(dst_class, img_name)
            )

        print(f"{cls}: {len(selected_imgs)} images")
        total_selected += len(selected_imgs)

    print(f"\n Dataset created at: {SRC_DRIVE}")
    print(f"Expected total images: {len(class_dirs) * IMAGES_PER_CLASS}")
    print(f"Actual total images: {total_selected}")

os.makedirs(OUT_SOLID_DRIVE, exist_ok=True)
os.makedirs(OUT_ENV_DRIVE, exist_ok=True)
os.makedirs(OUT_FLIP_DRIVE, exist_ok=True)
os.makedirs(TMP_ROOT, exist_ok=True)

print("Output folders ready")

In [ ]:
def parse_color(color_value):
    color_value = str(color_value).strip()
    if color_value.startswith("#") and len(color_value) == 7:
        return tuple(int(color_value[i:i+2], 16) for i in (1, 3, 5))
    return (255, 255, 255)

def list_env_backgrounds():
    if not os.path.isdir(ENV_BG_DIR):
        return []
    return sorted([
        os.path.join(ENV_BG_DIR, f)
        for f in os.listdir(ENV_BG_DIR)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

def remove_background_and_crop_fast(pil_img, session, padding=20):
    """
    Faster version using a persistent rembg session.
    """
    pil_img = pil_img.convert("RGB")

    rgba = remove(pil_img, session=session).convert("RGBA")

    alpha = rgba.getchannel("A")
    bbox = alpha.getbbox()

    if bbox is None:
        w, h = pil_img.size
        crop_w, crop_h = int(w * 0.65), int(h * 0.65)
        left = (w - crop_w) // 2
        top = (h - crop_h) // 2
        return pil_img.crop((left, top, left + crop_w, top + crop_h)).convert("RGBA")

    x1, y1, x2, y2 = bbox
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(rgba.width, x2 + padding)
    y2 = min(rgba.height, y2 + padding)

    return rgba.crop((x1, y1, x2, y2))

def paste_on_solid_background(fg_rgba, bg_hex="#00FF00", out_size=(224, 224), pad=0.08):
    color_rgb = parse_color(bg_hex)
    bg = Image.new("RGB", out_size, color_rgb)

    W, H = out_size
    max_w = int(W * (1 - 2 * pad))
    max_h = int(H * (1 - 2 * pad))

    fw, fh = fg_rgba.size
    scale = min(max_w / fw, max_h / fh)
    new_w = max(1, int(fw * scale))
    new_h = max(1, int(fh * scale))

    fg_resized = fg_rgba.resize((new_w, new_h), Image.LANCZOS)

    x = (W - new_w) // 2
    y = (H - new_h) // 2

    bg_rgba = bg.convert("RGBA")
    bg_rgba.alpha_composite(fg_resized, (x, y))
    return bg_rgba.convert("RGB")

def paste_on_environment_background(fg_rgba, env_path, out_size=(224, 224), pad=0.08):
    bg = Image.open(env_path).convert("RGB").resize(out_size)

    W, H = out_size
    max_w = int(W * (1 - 2 * pad))
    max_h = int(H * (1 - 2 * pad))

    fw, fh = fg_rgba.size
    scale = min(max_w / fw, max_h / fh)
    new_w = max(1, int(fw * scale))
    new_h = max(1, int(fh * scale))

    fg_resized = fg_rgba.resize((new_w, new_h), Image.LANCZOS)

    x = (W - new_w) // 2
    y = (H - new_h) // 2

    bg_rgba = bg.convert("RGBA")
    bg_rgba.alpha_composite(fg_resized, (x, y))
    return bg_rgba.convert("RGB")

def hand_flip_on_white(pil_img, out_size=(224, 224)):
    return ImageOps.mirror(pil_img.convert("RGB")).resize(out_size, Image.LANCZOS)

In [ ]:
print("Loading rembg session:")
session = new_session("u2net")
print("rembg session ready")

In [ ]:
def process_class_folder_fast(class_name, env_paths, session):
    src_class_drive = os.path.join(SRC_DRIVE, class_name)
    tmp_class_local = os.path.join(TMP_ROOT, class_name)

    out_solid_class = os.path.join(OUT_SOLID_DRIVE, class_name)
    out_env_class   = os.path.join(OUT_ENV_DRIVE, class_name)
    out_flip_class  = os.path.join(OUT_FLIP_DRIVE, class_name)

    os.makedirs(out_solid_class, exist_ok=True)
    os.makedirs(out_env_class, exist_ok=True)
    os.makedirs(out_flip_class, exist_ok=True)

    if os.path.exists(tmp_class_local):
        shutil.rmtree(tmp_class_local)

    shutil.copytree(src_class_drive, tmp_class_local)

    images = [f for f in os.listdir(tmp_class_local) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    for fname in tqdm(images, desc=f"FAST processing {class_name}"):
        local_img_path = os.path.join(tmp_class_local, fname)

        try:
            img = Image.open(local_img_path).convert("RGB")

            #Hand flip
            flip_img = hand_flip_on_white(img, out_size=OUT_SIZE)
            flip_img.save(os.path.join(out_flip_class, fname), quality=95)

            #background removal
            fg_crop = remove_background_and_crop_fast(img, session=session, padding=20)

            #solid background
            solid_color = random.choice(SOLID_COLORS)
            solid_img = paste_on_solid_background(fg_crop, bg_hex=solid_color, out_size=OUT_SIZE)
            solid_img.save(os.path.join(out_solid_class, fname), quality=95)

            #environment background
            if env_paths:
                env_path = random.choice(env_paths)
                env_img = paste_on_environment_background(fg_crop, env_path, out_size=OUT_SIZE)
            else:
                env_img = paste_on_solid_background(fg_crop, bg_hex="#FFFFFF", out_size=OUT_SIZE)

            env_img.save(os.path.join(out_env_class, fname), quality=95)

        except Exception as e:
            print(f"Error processing {class_name}/{fname}: {e}")

    shutil.rmtree(tmp_class_local)

In [ ]:
env_paths = list_env_backgrounds()
print("Environment backgrounds found:", len(env_paths))

class_dirs = [d for d in sorted(os.listdir(SRC_DRIVE)) if os.path.isdir(os.path.join(SRC_DRIVE, d))]
print("Classes found:", class_dirs)
print("Total classes found:", len(class_dirs))

for cls in class_dirs:
    process_class_folder_fast(cls, env_paths, session)

print("\nAll classes processed successfully.")

In [ ]:
print("Solid background dataset:")
!find "$OUT_SOLID_DRIVE" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l

print("Environment background dataset:")
!find "$OUT_ENV_DRIVE" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l

print("Hand flip dataset:")
!find "$OUT_FLIP_DRIVE" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l

In [ ]:
#Displaying one image from each modified test sets
from IPython.display import display

sample_class = class_dirs[0]
sample_files = os.listdir(os.path.join(SRC_DRIVE, sample_class))[:2]

print("Class:", sample_class)

for f in sample_files:
    print("File:", f)

    orig = Image.open(os.path.join(SRC_DRIVE, sample_class, f)).convert("RGB")
    solid = Image.open(os.path.join(OUT_SOLID_DRIVE, sample_class, f)).convert("RGB")
    env = Image.open(os.path.join(OUT_ENV_DRIVE, sample_class, f)).convert("RGB")
    flip = Image.open(os.path.join(OUT_FLIP_DRIVE, sample_class, f)).convert("RGB")

    display(orig.resize((224, 224)))
    display(solid)
    display(env)
    display(flip)